In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.14 The Angular-Momentum Algebra

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.14",
    title="The Angular-Momentum Algebra",
    blurb="Rotations, by algebra alone. The three components of angular momentum "
    "refuse to commute, and from that single fact — with the same ladder trick that "
    "solved the oscillator — the entire spectrum of allowed angular momenta falls "
    "out: the quantized magnitudes, the discrete orientations, and a surprise the "
    "classical world never hinted at. The ladder must close, and closing it forces "
    "half-integer spin into existence. The qubit we began with turns out to be the "
    "smallest rotation there is.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

Movement III opens by going back to the most elegant idea in the volume and using it again. In [§6.12](harmonic-oscillator.ipynb) the
harmonic oscillator gave up its entire spectrum to a pair of **ladder operators**, with not a single
derivative taken: from the one commutator $[a,a^{\dagger}]=1$, the equally-spaced levels followed by
pure algebra. This notebook applies exactly that method to **rotations**, and the result is richer than
the oscillator's.

Angular momentum in quantum mechanics is not a formula to be looked up. It is *defined* by an algebra —
the three components $J_x,J_y,J_z$ satisfy $[J_i,J_j]=i\hbar\,\varepsilon_{ijk}J_k$, the quantum image
of the everyday fact that rotations about different axes do not commute (turn a book about $x$ then $y$,
then reverse the order: it ends up facing differently). Because the components do not commute, they are
**incompatible** in the precise sense of [§6.6](pauli-uncertainty.ipynb) — no state can have all three sharp at once. But there is a
special combination, the total $J^2=J_x^2+J_y^2+J_z^2$, that *does* commute with each component, so the
magnitude and **one** projection (conventionally $J_z$) can be simultaneously definite. Those shared
eigenstates are the $|j,m\rangle$, and finding their spectrum is the whole game.

We play it with the **ladder operators** $J_\pm=J_x\pm iJ_y$, which raise and lower $m$ exactly as
$a^{\dagger},a$ raised and lowered the oscillator's quantum number. The ladder cannot run forever — the
projection $m$ is bounded by the magnitude — so it must terminate at a top and a bottom rung, and
*forcing it to close* fixes everything: $J^2=\hbar^2 j(j+1)$, $J_z=\hbar m$ with $m$ running from $-j$
to $+j$ in integer steps, hence $2j+1$ rungs. And here is the surprise the oscillator never sprang on
us: $2j+1$ must be a positive integer, so $j$ can be a **half-integer** — $j=\tfrac12,\tfrac32,\dots$ —
values no orbiting particle can have. The smallest of them, $j=\tfrac12$, turns out to reproduce the
Pauli matrices exactly: the **spin-$\tfrac12$ qubit** of Movement I ([§6.4](stern-gerlach-qubit.ipynb)–[§6.8](bloch-sphere-entanglement.ipynb)) is not an extra postulate
bolted onto quantum mechanics. It is the smallest nonzero angular momentum, and the algebra of rotations
demanded its existence.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the explicit construction of $J_\pm$ from the matrix elements
$\hbar\sqrt{j(j+1)-m(m\pm1)}$, then $J_x=(J_++J_-)/2$, $J_y=(J_+-J_-)/2i$, $J_z=\hbar\,$`numpy.diag(m)`,
the spectra read with `numpy.linalg.eigvalsh`, and every commutator a matrix product `A@B - B@A`.

> **Conventions and method notes.** We keep $\hbar$ explicit (set to $1$ numerically, but written in
> every formula). For a given $j$ the basis is $|j,m\rangle$ with $m$ ordered **descending**,
> $m=j,j-1,\dots,-j$, so the matrices are $(2j+1)\times(2j+1)$ and $J_z=\hbar\,\mathrm{diag}(j,\dots,-j)$.
> The raising and lowering operators $J_\pm$ are **non-Hermitian** and mutual adjoints
> ($J_-=J_+^{\dagger}$); the components $J_x,J_y,J_z$ and $J^2$ are Hermitian. See Sakurai &
> Napolitano (§3.5–3.6, the algebraic construction); Nolting (angular momentum); and Notebooks [§6.6](pauli-uncertainty.ipynb) (the
> spin algebra, incompatibility, compatible observables), [§6.12](harmonic-oscillator.ipynb) (the ladder method for the oscillator),
> [§6.2](operators-spectral-theorem.ipynb) (commuting operators share an eigenbasis), and [§6.4](stern-gerlach-qubit.ipynb)–[§6.8](bloch-sphere-entanglement.ipynb) (the qubit, here revealed as spin-$\tfrac12$).

## Theory in brief

### The defining algebra

Angular momentum is *any* triple of Hermitian operators obeying the commutation relations

```{math}
:label: eq-angmom-algebra
[J_i,J_j]=i\hbar\,\varepsilon_{ijk}J_k\qquad\Longleftrightarrow\qquad [J_x,J_y]=i\hbar J_z,\quad [J_y,J_z]=i\hbar J_x,\quad [J_z,J_x]=i\hbar J_y .
```

This generalizes the spin algebra $[S_x,S_y]=i\hbar S_z$ of [§6.6](pauli-uncertainty.ipynb) and is the quantum image of the
classical fact that finite rotations about different axes do not commute. Because the components do not
commute, they are **incompatible** ([§6.6](pauli-uncertainty.ipynb)): no state can have $J_x,J_y,J_z$ all sharp at once.

### The Casimir operator $J^2$

The total squared angular momentum

```{math}
:label: eq-casimir
J^2=J_x^2+J_y^2+J_z^2,\qquad [J^2,J_i]=0\ \ \text{for every }i,
```

**commutes** with each component (a few lines from {eq}`eq-angmom-algebra`). So $J^2$ and one component
— take $J_z$ — form a **compatible pair** ([§6.6](pauli-uncertainty.ipynb)) and share an eigenbasis: the simultaneous eigenstates
$|j,m\rangle$, labelled by the eigenvalues of $J^2$ and $J_z$. ($J^2$ is a *Casimir operator*: constant
across each irreducible rotation multiplet.)

### The ladder operators

Define the **raising** and **lowering** operators

```{math}
:label: eq-angmom-ladder
J_\pm=J_x\pm iJ_y,\qquad [J_z,J_\pm]=\pm\hbar J_\pm,\qquad [J^2,J_\pm]=0 .
```

They are non-Hermitian and mutual adjoints. The commutator $[J_z,J_\pm]=\pm\hbar J_\pm$ says $J_+$
**raises** the $J_z$ eigenvalue by $\hbar$ and $J_-$ **lowers** it: acting on $|j,m\rangle$,
$J_\pm|j,m\rangle\propto|j,m\pm1\rangle$, while $[J^2,J_\pm]=0$ keeps $j$ fixed. This is exactly the
oscillator's $a^{\dagger}/a$ method ([§6.12](harmonic-oscillator.ipynb)), now climbing a ladder of orientations instead of energies.

### The spectrum from the algebra

Because $J^2-J_z^2=J_x^2+J_y^2\ge0$, the projection $m$ is **bounded**, so the ladder must terminate:
there is a top rung with $J_+|j,j\rangle=0$ and a bottom with $J_-|j,-j\rangle=0$. Working out the norms
of $J_\pm|j,m\rangle$ and demanding closure gives the entire spectrum,

```{math}
:label: eq-angmom-spectrum
J^2|j,m\rangle=\hbar^2 j(j+1)\,|j,m\rangle,\qquad J_z|j,m\rangle=\hbar m\,|j,m\rangle,\qquad J_\pm|j,m\rangle=\hbar\sqrt{j(j+1)-m(m\pm1)}\;|j,m\pm1\rangle,
```

with $m=-j,-j+1,\dots,+j$ — exactly $2j+1$ values. Note the magnitude $\sqrt{j(j+1)}\,\hbar$ **exceeds**
the largest projection $j\hbar$: angular momentum can never point fully along an axis, for if it did all
three components would be sharp, violating {eq}`eq-angmom-algebra`. This is the *vector model* — a
tilted vector precessing about the axis it most nearly aligns with (Exercise 7).

### Integer and half-integer $j$

The ladder runs from $-j$ to $+j$ in unit steps, so it holds $2j+1$ rungs, and

```{math}
:label: eq-half-integer
2j+1\in\mathbb{Z}^{+}\quad\Longrightarrow\quad j\in\{0,\tfrac12,1,\tfrac32,2,\dots\} .
```

The **integer** values are *orbital* angular momentum ([§6.15](orbital-angular-momentum.ipynb)). The **half-integer** values have no
orbital realization — they are **spin**. The smallest nonzero case, $j=\tfrac12$, reproduces the Pauli
matrices: $J_x=\sigma_x/2,\;J_y=\sigma_y/2,\;J_z=\sigma_z/2$ (in units $\hbar=1$). So the two-state
system of Movement I **is** spin-$\tfrac12$ angular momentum, derived here from rotations rather than
postulated. (Why half-integers need the rotation group's double cover — the $4\pi$ periodicity of
spinors glimpsed on the Bloch sphere of [§6.8](bloch-sphere-entanglement.ipynb) — is a horizon flagged in the Outlook.)

### Constructing the matrices for any $j$

With the matrix elements of {eq}`eq-angmom-spectrum` we build, for any $j$,

```{math}
:label: eq-angmom-matrices
J_z=\hbar\,\mathrm{diag}(j,\dots,-j),\quad (J_+)_{m+1,\,m}=\hbar\sqrt{j(j+1)-m(m+1)},\quad J_x=\tfrac12(J_++J_-),\quad J_y=\tfrac1{2i}(J_+-J_-),
```

explicit $(2j+1)\times(2j+1)$ matrices — the reusable building blocks for spin systems ([§6.18](spin-magnetic.ipynb)), the
addition of angular momenta ([§6.19](addition-angular-momenta.ipynb)), and rotations of quantum states.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Arc

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0  # written explicitly in every formula; set to 1 for the numerics


def angular_momentum_matrices(j, hbar=HBAR):
    r"""Build the angular-momentum matrices for a given $j$ in the $|j,m\rangle$ basis {eq}`eq-angmom-matrices`.

    The basis is ordered with $m$ **descending** ($m=j,j-1,\dots,-j$), so the matrices are
    $(2j+1)\times(2j+1)$. The raising/lowering operators are filled directly from the matrix elements
    $J_\pm|j,m\rangle=\hbar\sqrt{j(j+1)-m(m\pm1)}\,|j,m\pm1\rangle$ {eq}`eq-angmom-spectrum`; then
    $J_x=(J_++J_-)/2$, $J_y=(J_+-J_-)/2i$, $J_z=\hbar\,$``numpy.diag``$(m)$, and
    $J^2=J_x^2+J_y^2+J_z^2$.

    Parameters
    ----------
    j : float
        The angular-momentum quantum number, integer or half-integer (e.g. ``0.5``, ``1``, ``1.5``).
    hbar : float, optional
        The reduced Planck constant (default ``1.0``).

    Returns
    -------
    Jx, Jy, Jz, J2, Jp, Jm : numpy.ndarray
        The Hermitian components ``Jx, Jy, Jz``, the Casimir ``J2``, and the non-Hermitian ladder
        operators ``Jp`` ($J_+$) and ``Jm`` ($J_-$), each a complex $(2j+1)\times(2j+1)$ array.
    """
    dim = int(round(2 * j + 1))
    m = np.arange(j, -j - 1, -1.0)  # descending: j, j-1, ..., -j
    Jz = hbar * np.diag(m).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    Jm = np.zeros((dim, dim), dtype=complex)
    for a in range(dim):
        ma = m[a]
        c_plus = j * (j + 1) - ma * (
            ma + 1
        )  # J+ : |j,m> -> |j,m+1>  (index a-1, descending)
        if c_plus > 1e-12:
            Jp[a - 1, a] = hbar * np.sqrt(c_plus)
        c_minus = j * (j + 1) - ma * (ma - 1)  # J- : |j,m> -> |j,m-1>  (index a+1)
        if c_minus > 1e-12:
            Jm[a + 1, a] = hbar * np.sqrt(c_minus)
    Jx = (Jp + Jm) / 2
    Jy = (Jp - Jm) / 2j  # 2i
    J2 = Jx @ Jx + Jy @ Jy + Jz @ Jz
    return Jx, Jy, Jz, J2, Jp, Jm


def commutator(A, B):
    """The commutator $[A,B]=AB-BA$, the matrix product ``A@B - B@A``."""
    return A @ B - B @ A


def m_values(j):
    """The descending ladder of projection quantum numbers $m=j,j-1,\\dots,-j$ (``numpy.arange``)."""
    return np.arange(j, -j - 1, -1.0)


def ket_index(j, m):
    """Index of the basis state $|j,m\\rangle$ in the descending convention (so $m=j\\to0$)."""
    return int(round(j - m))

## Exercise 1 — The angular-momentum algebra

Construct the angular-momentum matrices for a given $j$ and verify the **defining commutation
relations** $[J_x,J_y]=i\hbar J_z$ and its cyclic partners {eq}`eq-angmom-algebra`. This algebra
*is* the definition of angular momentum; everything else in the notebook follows from it.

1. Build $J_x,J_y,J_z$ for, say, $j=1$ with the `angular_momentum_matrices` helper (which fills
   $J_\pm$ from $\hbar\sqrt{j(j+1)-m(m\pm1)}$ and forms $J_x=(J_++J_-)/2$, $J_y=(J_+-J_-)/2i$,
   $J_z=\hbar\,$`numpy.diag`).
2. Compute the three commutators $[J_x,J_y]$, $[J_y,J_z]$, $[J_z,J_x]$ with the `commutator`
   helper (each a matrix product `A@B - B@A`).
3. Confirm they equal $i\hbar J_z$, $i\hbar J_x$, $i\hbar J_y$ with `numpy.allclose`.
4. Note this is the same algebra as the spin $[S_x,S_y]=i\hbar S_z$ of [§6.6](pauli-uncertainty.ipynb), now for arbitrary $j$.
   Frame: the algebra defines angular momentum.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    algebra_holds,
    "the constructed matrices satisfy the angular-momentum algebra [Jx,Jy]=iℏJz and cyclic — the definition of angular momentum",
)

## Exercise 2 — The Casimir operator $J^2$

Show that the total squared angular momentum $J^2=J_x^2+J_y^2+J_z^2$ **commutes with every
component**, so $J^2$ and $J_z$ are a compatible pair and share the $|j,m\rangle$ eigenbasis
{eq}`eq-casimir`.

1. Form $J^2=J_x^2+J_y^2+J_z^2$ by matrix products (`Jx@Jx + Jy@Jy + Jz@Jz`, already returned by
   the helper).
2. Compute $[J^2,J_z]$ and $[J^2,J_x]$ with the `commutator` helper.
3. Confirm both vanish with `numpy.allclose` against the zero matrix.
4. Conclude ([§6.2](operators-spectral-theorem.ipynb), [§6.6](pauli-uncertainty.ipynb)): because they commute, $J^2$ and $J_z$ can be simultaneously diagonalized —
   the shared eigenstates are the $|j,m\rangle$. Frame: the magnitude and one projection can be
   sharp together, but not two projections.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array(
        [
            np.linalg.norm(casimir_xy),
            np.linalg.norm(casimir_y),
            np.linalg.norm(casimir_z),
        ]
    ),
    np.zeros(3),
    "J² commutes with each component ([J²,Jᵢ]=0), so J² and Jz share the |j,m⟩ eigenbasis",
    atol=1e-12,
)

## Exercise 3 — The ladder operators

Build the raising and lowering operators $J_\pm=J_x\pm iJ_y$ and verify they **raise and lower**
$m$ by one unit, with the coefficient $\hbar\sqrt{j(j+1)-m(m\pm1)}$, and that they **annihilate**
the top and bottom rungs {eq}`eq-angmom-ladder`, {eq}`eq-angmom-spectrum`.

1. Take $J_\pm=J_x\pm iJ_y$ (returned by the helper as `Jp`, `Jm`).
2. Verify the ladder commutators $[J_z,J_\pm]=\pm\hbar J_\pm$ and $[J^2,J_\pm]=0$ with the
   `commutator` helper and `numpy.allclose`.
3. Apply $J_+$ to a chosen $|j,m\rangle$ (a unit basis vector) and read off the result: it lands
   on $|j,m+1\rangle$ with coefficient $\hbar\sqrt{j(j+1)-m(m+1)}$.
4. Confirm $J_+|j,j\rangle=0$ (top) and $J_-|j,-j\rangle=0$ (bottom): the ladder terminates.
   Frame: the oscillator's ladder method ([§6.12](harmonic-oscillator.ipynb)), now climbing orientations.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    ladder_works,
    "J± raise/lower m by one unit with coefficient ℏ√(j(j+1)−m(m±1)), satisfy [Jz,J±]=±ℏJ± and [J²,J±]=0, and annihilate the top/bottom rungs",
)

## Exercise 4 — The spectrum from the ladder

Read off the eigenvalues of $J^2$ and $J_z$ and confirm the algebra's central predictions: a
single magnitude $J^2=\hbar^2 j(j+1)$ and an evenly-spaced ladder of projections $J_z=\hbar m$,
$m=-j,\dots,+j$, exactly $2j+1$ of them {eq}`eq-angmom-spectrum`. This is the payoff — the entire
spectrum from the commutation relations alone, no coordinates and no differential equation.

1. For each $j\in\{\tfrac12,1,\tfrac32,2\}$ build the matrices and diagonalize $J^2$ with
   `numpy.linalg.eigvalsh` — find the *single* repeated eigenvalue $\hbar^2 j(j+1)$.
2. Diagonalize $J_z$ the same way (or read its diagonal): the eigenvalues are $\hbar m$,
   $m=-j,\dots,j$.
3. Confirm there are $2j+1$ of them.
4. Note $\sqrt{j(j+1)}>j$: the magnitude exceeds the maximum projection, so the vector cannot
   align fully with any axis. Frame: the spectrum from the algebra alone.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
# collect the j=2 spectrum for a concrete numerical check
Jx2, Jy2, Jz2, J22, _, _ = angular_momentum_matrices(2.0)
validate.close(
    np.concatenate(
        [[np.linalg.eigvalsh(J22)[0]], np.sort(np.linalg.eigvalsh(Jz2))[::-1]]
    ),
    np.concatenate([[HBAR**2 * 2 * 3], HBAR * m_values(2.0)]),
    "the spectrum follows from the algebra: J²=ℏ²j(j+1) and Jz=ℏm with m=−j…j (2j+1 states)",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Integer and half-integer $j$

Explain why $j$ must be **integer or half-integer**, and exhibit the half-integer case: show that
$j=\tfrac12$ reproduces the Pauli matrices over two, so the qubit of Movement I **is** spin-
$\tfrac12$, the smallest nonzero angular momentum {eq}`eq-half-integer`.

1. Recall the counting argument: the ladder steps from $-j$ to $+j$ in unit steps, so it has
   $2j+1$ rungs, and a count of rungs must be a positive integer — hence $2j+1\in\mathbb{Z}^+$ and
   $j\in\{0,\tfrac12,1,\tfrac32,\dots\}$.
2. Build the $j=\tfrac12$ matrices with `angular_momentum_matrices(0.5)`.
3. Compare to the Pauli matrices over two ($\sigma_x/2,\sigma_y/2, \sigma_z/2$) with
   `numpy.allclose`.
4. Conclude: the two-state system of [§6.4](stern-gerlach-qubit.ipynb)–[§6.8](bloch-sphere-entanglement.ipynb) is spin-$\tfrac12$ angular momentum, derived from the
   algebra rather than postulated; the integer $j$ are orbital ([§6.15](orbital-angular-momentum.ipynb)). Frame: half-integer spin
   emerges from the closure of the ladder — the surprise of the notebook.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    matches_pauli and counts_are_integers,
    "ladder closure forces 2j+1 to be a positive integer (so j is integer or half-integer), and j=½ reproduces the Pauli matrices over two — the spin-½ qubit",
)

## Exercise 6 — Matrices for arbitrary $j$: a reusable tool

Assemble the full set of angular-momentum matrices for a chosen $j$ and verify their **internal
consistency** — the algebra, Hermiticity, the adjoint relation $J_-=J_+^{\dagger}$, and
$J^2=\hbar^2 j(j+1)\mathbb{1}$ — confirming the construction is correct for any $j$
{eq}`eq-angmom-matrices`.

1. For a half-integer and an integer case (say $j=\tfrac32$ and $j=2$) build
   $J_x,J_y,J_z,J^2,J_\pm$ with the helper.
2. Verify the algebra $[J_x,J_y]=i\hbar J_z$ and $J^2=\hbar^2 j(j+1)\mathbb{1}$ (`numpy.allclose`
   against `j*(j+1)*numpy.eye`).
3. Confirm $J_x,J_y,J_z$ are Hermitian (`numpy.allclose(A, A.conj().T)`) and that
   $J_-=J_+^{\dagger}$.
4. Note these matrices are the reusable building blocks for spin in fields ([§6.18](spin-magnetic.ipynb)) and the addition
   of angular momenta ([§6.19](addition-angular-momenta.ipynb)). Frame: a general-purpose construction, correct for every $j$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    all_consistent,
    "the angular-momentum matrices can be built for any j: they satisfy the algebra, are Hermitian (J± mutual adjoints), and give J²=ℏ²j(j+1)",
)

## Exercise 7 — Expectation values, uncertainty, and the vector model *(student)*

In the maximally-aligned state $|j,j\rangle$ (top of the ladder, $J_z$ as sharp as it can be),
compute $\langle J_x\rangle,\langle J_y\rangle,\langle J_x^2\rangle,\langle J_y^2\rangle$ and the
**uncertainty product** $\Delta J_x\,\Delta J_y$, and interpret the result geometrically as the
*vector model* {eq}`eq-angmom-spectrum`, {eq}`eq-angmom-algebra`.

1. Build the $|j,j\rangle$ state (the first basis vector in the descending convention) for, say,
   $j=2$.
2. Compute the expectation values $\langle O\rangle=\psi^{\dagger}O\psi$: find $\langle
   J_x\rangle=\langle J_y\rangle=0$ but $\langle J_x^2\rangle=\langle J_y^2\rangle=\hbar^2 j/2>0$.
3. Form the uncertainties $\Delta J_x=\sqrt{\langle J_x^2\rangle-\langle J_x\rangle^2}$ and
   likewise $\Delta J_y$, and check the [§6.6](pauli-uncertainty.ipynb) uncertainty relation $\Delta J_x\,\Delta
   J_y\ge\tfrac12|\langle J_z\rangle|$ — it is **saturated** here.
4. Interpret: $J_z$ is sharp at $j\hbar$, but $J_x,J_y$ are spread, so the angular-momentum vector
   of length $\sqrt{j(j+1)}\,\hbar$ cannot point straight up — it lies on a **cone**, precessing
   about $z$. Frame: the vector model, and why a quantum spin never fully aligns.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    np.isclose(exp_Jx, 0)
    and np.isclose(exp_Jy, 0)
    and exp_Jx2 > 0
    and np.isclose(exp_Jx2, HBAR**2 * j / 2)
    and np.isclose(dJx * dJy, bound),
    "even the most aligned state |j,j⟩ has ⟨Jx⟩=⟨Jy⟩=0 with nonzero transverse spread ⟨Jx²⟩=ℏ²j/2, saturating ΔJxΔJy≥½|⟨Jz⟩| — the vector model",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The structure of rotations, and the origin of spin

We wrote down three operators that refuse to commute — nothing more than $[J_i,J_j]=i\hbar\,
\varepsilon_{ijk}J_k$ — and from that single fact, with the same ladder that solved the oscillator in
[§6.12](harmonic-oscillator.ipynb), the entire spectrum of angular momentum fell out: magnitudes $\hbar\sqrt{j(j+1)}$, projections
$\hbar m$, and the demand that the ladder *close*, which forced $j$ to be integer or half-integer. We
never integrated anything; we never wrote a wave function. The algebra alone fixed what rotations are
allowed to do.

The half-integers were the surprise. They have no orbital meaning — you cannot build $j=\tfrac12$ from a
particle going around a loop — yet the algebra insisted on them, and the smallest, $j=\tfrac12$,
reproduced the Pauli matrices exactly. The two-state system we spent all of Movement I on is therefore
not a separate piece of physics: it **is** the smallest nonzero angular momentum, spin-$\tfrac12$,
derived here rather than postulated.

**This exercise (synthesis).** No new computation: the structure is the result. Movement III now splits
along the seam this notebook exposed. The next notebook ([§6.15](orbital-angular-momentum.ipynb)) builds the **integer** case as *orbital*
angular momentum — the spherical harmonics that shape every atomic orbital, realized as functions on the
sphere — reusing the real-spherical-harmonic machinery of Volume III. Later we return to spin in magnetic
fields ([§6.18](spin-magnetic.ipynb)) and to the *addition* of two angular momenta ([§6.19](addition-angular-momenta.ipynb)), and the two strands — orbital and spin
— braid together at the summit of the volume, the hydrogen atom ([§6.17](hydrogen-atom.ipynb)), where $J^2$ and $J_z$ label
every orbital.

There is no picture you can draw of a spin-$\tfrac12$ particle's rotation: turn it through $360^\circ$
and it is *not* the same state — you must turn it twice, through $720^\circ$, to bring it home (the
$4\pi$ periodicity glimpsed on the Bloch sphere of [§6.8](bloch-sphere-entanglement.ipynb)). We did not assume this. The algebra of
rotations, taken seriously, produced a kind of angular momentum the spinning world around us never
shows — and matter, every electron and proton and neutron in it, is made of exactly this.

## Notebook summary

Angular momentum, defined by an algebra and solved by a ladder — the opening of Movement III.

- **The defining algebra** {eq}`eq-angmom-algebra`: $[J_i,J_j]=i\hbar\,\varepsilon_{ijk}J_k$. The
  components are incompatible ([§6.6](pauli-uncertainty.ipynb)); this *is* the definition of angular momentum.
- **The Casimir** {eq}`eq-casimir`: $J^2$ commutes with every component, so $J^2$ and $J_z$ are
  compatible and share the $|j,m\rangle$ eigenbasis.
- **The ladder** {eq}`eq-angmom-ladder`: $J_\pm=J_x\pm iJ_y$ with $[J_z,J_\pm]=\pm\hbar J_\pm$ raise and
  lower $m$ — the oscillator's method ([§6.12](harmonic-oscillator.ipynb)), now for rotations.
- **The spectrum** {eq}`eq-angmom-spectrum`: from closure alone, $J^2=\hbar^2 j(j+1)$ and $J_z=\hbar m$,
  $m=-j,\dots,j$ — and $\sqrt{j(j+1)}>j$ (the vector model cone).
- **Integer and half-integer $j$** {eq}`eq-half-integer`: $2j+1\in\mathbb{Z}^+$, so half-integers are
  allowed — and $j=\tfrac12$ **is** the Pauli/2 qubit of Movement I, derived.
- **A reusable construction** {eq}`eq-angmom-matrices`: explicit $(2j+1)\times(2j+1)$ matrices for any
  $j$ (`numpy.diag`, the $J_\pm$ matrix elements, `numpy.linalg.eigvalsh`), the building blocks for spin
  and angular-momentum coupling.

Three non-commuting operators, one ladder, and out came every allowed angular momentum — including the
half-integer spin the classical world never hinted at. The qubit was the smallest rotation all along.

## Outlook

- **Orbital angular momentum and the spherical harmonics ([§6.15](orbital-angular-momentum.ipynb))**: the integer-$j$ case, realized as
  functions on the sphere, reusing the Volume III real-spherical-harmonic machinery.
- **The hydrogen atom ([§6.17](hydrogen-atom.ipynb))**: where $J^2$ and $J_z$ label the orbitals and the algebra meets the
  Coulomb potential.
- **Spin in magnetic fields ([§6.18](spin-magnetic.ipynb))** and **the addition of two angular momenta** with Clebsch–Gordan
  coefficients ([§6.19](addition-angular-momenta.ipynb)) — both built directly on the matrices of this notebook.
- **The double cover of the rotation group** and the $4\pi$ periodicity of spinors (a horizon; the [§6.8](bloch-sphere-entanglement.ipynb)
  Bloch double-cover made this visible).
- **Cross-reference** [§6.6](pauli-uncertainty.ipynb) (the spin algebra, incompatibility, the Casimir/compatible observables), [§6.12](harmonic-oscillator.ipynb)
  (the ladder method for the oscillator), [§6.2](operators-spectral-theorem.ipynb) (commuting operators share an eigenbasis), [§6.4](stern-gerlach-qubit.ipynb)–[§6.8](bloch-sphere-entanglement.ipynb) (the
  qubit, now derived as spin-$\tfrac12$), and forward to [§6.15](orbital-angular-momentum.ipynb), [§6.17](hydrogen-atom.ipynb), [§6.18](spin-magnetic.ipynb), [§6.19](addition-angular-momenta.ipynb).

In [ ]:
from ecp.style import footer

footer()